In [5]:
import sys
sys.path.insert(0, "/workspaces/MyStudio")

from DEM_MCM.src.bucket_io import load_experiment_from_bucket
from DEM_MCM.src.analyze_results import MarkovAnalyzer

On constate que la matrice de transiton évolue avec le learning time. Autrement dit, elle évolue avec le temps. Ce qui confirme l'hypothèse selon laquelle elle n'est pas homogène . D'où son inappropriation pour le cas d'étude de la ségrégation presente.

Le découpage cartésien donne de très mauvais résultats sur la construction de la matrice de transition du fait qu'il n'établi pas de recouvrement qui qui capture toutes les particules. En effet, avec le découpage cartésien, il est difficile que toutes les partitions capturent les particules. De ce fait, il y a beaucoup de **nan** dans la matrice de transition. 

In [6]:
Matrix=load_experiment_from_bucket("cartesian_nx5_ny5_nz5_NLT100_step1_start250_dt0.5")
print(Matrix["matrix"].sum(axis=0))

[1.00000001 0.99889881 0.99863044 0.99873677 1.00000001 0.99750001
 0.99655431 0.99867742 0.99922039 1.00000001        nan        nan
 0.98758334 0.9983299  0.9995            nan        nan        nan
        nan        nan        nan        nan        nan        nan
        nan 1.00000001 0.99944445 1.00000001 1.00000001 1.00000001
 0.99091667 0.99583257 0.99833184 0.99964286 0.99846591        nan
        nan 0.9837381  0.99856411 0.99796782        nan        nan
        nan        nan 0.98107143        nan        nan        nan
        nan        nan        nan 0.99777908 0.99861068 1.00000001
 1.00000001        nan 0.99798137 0.99822123 0.99883616 0.99870446
        nan        nan 0.97983334 0.99665502 0.99901914        nan
        nan        nan        nan        nan        nan        nan
        nan        nan        nan 1.00000001 1.         0.99952381
 0.99911067 1.00000001 0.99300001 0.99555921 0.99888095 1.
 0.99947369        nan        nan 0.98083334 0.99805557 0.99856927
   

In [7]:
print(Matrix["stats"])

{'n_timesteps_used': 100, 'n_states': 125, 'n_states_visited': 63, 'n_states_empty': 62, 'fraction_visited': 0.504, 'column_sum_min': 0.979833344221115, 'column_sum_max': 1.0000000089406966, 'column_sum_mean': 0.9970650463394585, 'diagonal_mean': 0.43651305452629924, 'diagonal_std': 0.3379710399224092, 'method': 'cartesian', 'step_size': 1, 'dt': 0.5, 'overlap_ratio': 0.5, 'n_paires_par_step': 2.0, 'plage_temporelle': 100, 'start_index': 250, 'first_pair': [250, 251], 'last_pair': [349, 350]}


Comme nous pouvons le voir au travers des statistiques ci-dessus, il y a beaucoup de cellules vides lors du découpage qui fait que la matrice n'est pas utilisable pour effectuer la prédicton. Surtout quand le découpage est trop fin(dans la zone haute où il n'y a presque pas de particules ). 

Nous constatons que pour un découpage soit utilisable nous devons quelque soit le type de découpage avoir dans la zone base environ une dizaine de particules par partition. De ce fait, nous devons déjà effectué un découpage differentié du fait que nous soyons dans la partie haute ou dans la partie base du mélangeur de telle sorte que dans la partie haute du mélangeur, le découpage soit grossier permettant de capturer au mieux les mouvements des particules et dans la partie basse du mélangeur que le découpage soit plus fin pour capturer plus en détail la cinétique des particules pour une prédiction qui soit conforme à la réalité(au modèle DEM).

C'est dans cette optique nous avons dans un premier temps fait une rédiscretisation du mélangeur en mode **adaptatif** qui, lors du découpage, tiend compte du fait que le mélangeur n'a pas la même distribution de particules dans la partie basse que dans la partie haute. Elle divise le mélangeur grossièrement dans la partie haute et plus finement dans la partie basse. La méthode de découpage de la partie basse est l'une des méthodes déjà implémentée aupavant: 
- cylindrique
- cartésienne
- voronoï
- physique
- quantile
- octree



In [8]:
Analyzer=MarkovAnalyzer()

In [9]:
Analyzer.load_method('cartesian')

   ✅ cartesian_nx10_ny10_nz10_NLT100_step1_start250_dt0.1: shape=(1000, 1000)
   ✅ cartesian_nx12_ny12_nz12_NLT100_step1_start250_dt0.1: shape=(1728, 1728)
   ✅ cartesian_nx15_ny15_nz15_NLT100_step1_start250_dt0.1: shape=(3375, 3375)
   ✅ cartesian_nx18_ny18_nz18_NLT100_step1_start250_dt0.1: shape=(5832, 5832)
   ✅ cartesian_nx20_ny20_nz20_NLT100_step1_start250_dt0.1: shape=(8000, 8000)
   ✅ cartesian_nx2_ny2_nz2_NLT100_step1_start250_dt0.1: shape=(8, 8)
   ✅ cartesian_nx3_ny3_nz3_NLT100_step1_start250_dt0.1: shape=(27, 27)
   ✅ cartesian_nx4_ny4_nz4_NLT100_step1_start250_dt0.1: shape=(64, 64)
   ✅ cartesian_nx5_ny5_nz5_NLT100_step10_start250_dt0.1: shape=(125, 125)
   ✅ cartesian_nx5_ny5_nz5_NLT100_step15_start250_dt0.1: shape=(125, 125)
   ✅ cartesian_nx5_ny5_nz5_NLT100_step1_start1000_dt0.1: shape=(125, 125)
   ✅ cartesian_nx5_ny5_nz5_NLT100_step1_start2000_dt0.1: shape=(125, 125)
   ✅ cartesian_nx5_ny5_nz5_NLT100_step1_start250_dt0.01: shape=(125, 125)
   ✅ cartesian_nx5_ny5_nz5_NL

Nous pouvons donc